# Final biomarker selection 

---

# ১. Expression Difference

প্রথমে দেখা হয়েছে:

```text id="zzhquv"
CKD vs Healthy
```

এ gene expression আলাদা কিনা।

---

## ব্যবহার করা হয়েছে:

```text id="c4c73t"
diff_mean_log_expr
```

Formula:

\Delta expression = \mu_{CKD} - \mu_{Control}

---

## Meaning

| Result      | Meaning     |
| ----------- | ----------- |
| বড় positive | CKD তে বেশি |
| বড় negative | CKD তে কম   |

---

# ২. Statistical Significance

Difference random কিনা check করা হয়েছে।

---

## ব্যবহার:

```text id="cykgrh"
p_value
adj_p_value
```

---

## Rule

```text id="8k1jlwm"
adj_p_value < 0.05
```

---

# ৩. Biological Effect Size

Difference biologically meaningful কিনা।

---

## ব্যবহার:

```text id="4vb3pn"
cohen_d
```

---

## বড় effect মানে:

```text id="1g6g0u"
CKD disease এ gene strongly involved
```

---

# ৪. Expression Percentage

Gene খুব কম cell এ থাকলে biomarker unreliable হতে পারে।

---

## ব্যবহার:

```text id="jlwm0d"
pct_expr_case
pct_expr_control
```

---

## Example

| Gene | CKD Cells |
| ---- | --------- |
| TLR7 | 85%       |

এটা strong marker হতে পারে।

---

# ৫. Expression Filter

Low-expression noisy gene remove করা হয়েছে।

---

## Filter

```python id="0o8wli"
min_pct = 0.05
min_mean = 0.01
```

---

# ৬. CKD-specific কিনা

সব kidney injury gene CKD-specific না।

তাই:

```text id="w5a93q"
CKD vs AKI
```

compare করা হয়েছে।

---

## ব্যবহার:

```text id="b8v9ep"
diff_mean_log_expr_CKD_vs_AKI
adj_p_value_CKD_vs_AKI
```

---

## যদি:

```text id="c0wx6m"
CKD >> AKI
```

তাহলে gene CKD-specific।

---

# ৭. Specificity Score

এটা measure করে gene কতটা CKD-specific।

---

## ব্যবহার:

```text id="owxrgw"
ckd_specificity_score
```

---

# ৮. Combined Biomarker Score

শেষে সব combine করা হয়েছে।

---

## Main Final Formula

\text{combined biomarker score}=\text{rank score}\times (1+\text{CKD specificity score})

---

# Rank Score এর ভিতরে কী আছে?

\text{rank score}= -\log_{10}(FDR) \times |Cohen's\ d| \times |\Delta expression|

---

# Meaning

Final biomarker score depend করছে:

✅ significance
✅ effect size
✅ expression difference
✅ CKD specificity

এর উপর।

---

# Machine Learning Support

এরপর Elastic Net model দেখেছে:

```text id="gcq97j"
এই gene disease predict করতে useful কিনা
```

---

## ব্যবহার:

```text id="q1y2iy"
elasticnet_gene_importance.csv
```

---

# Final Biomarker আসলে কেমন?

একটা ideal biomarker:

| Feature                   | Needed |
| ------------------------- | ------ |
| CKD তে বেশি/কম            | ✅      |
| statistically significant | ✅      |
| বড় effect size            | ✅      |
| অনেক cell এ expressed     | ✅      |
| AKI তে না                 | ✅      |
| ML model important বলছে   | ✅      |

---

# পুরো Decision Pipeline

```text id="jlwmrx"
Gene Expression
      ↓
CKD vs Healthy Difference
      ↓
Statistical Significance
      ↓
Effect Size
      ↓
Expression Percentage
      ↓
Expression Filter
      ↓
CKD vs AKI Specificity
      ↓
Combined Biomarker Score
      ↓
Machine Learning Validation
      ↓
Final Biomarker
```



# প্রথম Formula

## Rank Score

\text{rank score}= -\log_{10}(FDR) \times |Cohen's\ d| \times |\Delta expression|

এখানে ৩টা জিনিস combine করা হয়েছে।

---

# ১. `−log10(FDR)`

## কেন?

এটা statistical confidence measure করে।

---

## FDR ছোট হলে:

| FDR     | −log10(FDR) |
| ------- | ----------- |
| 0.05    | 1.3         |
| 0.001   | 3           |
| 0.00001 | 5           |

---

## Meaning

FDR যত ছোট:

```text id="iy5mjlwm"
gene তত reliable
```

---

## কেন log নেওয়া হয়েছে?

কারণ p-value খুব ছোট হতে পারে।

যেমন:

```text id="xhmq9m"
0.0000001
```

এগুলো directly compare করা কঠিন।

তাই:

```text id="97vw4g"
-log10()
```

নিয়ে scale করা হয়েছে।

---

# ২. `|Cohen’s d|`

## কেন?

p-value ছোট হলেই important না।

কারণ:

* বড় dataset এ tiny difference ও significant হতে পারে

তাই biological effect size দরকার।

---

## Cohen’s d বলে:

```text id="4qdd4m"
difference কত বড়
```

---

## Example

| Gene   | p-value | Effect |
| ------ | ------- | ------ |
| Gene A | tiny    | huge   |
| Gene B | tiny    | tiny   |

এখানে Gene A বেশি useful biomarker।

---

## Absolute নেওয়া হয়েছে কেন?

কারণ:

* upregulated
* downregulated

দুইটাই important হতে পারে।

তাই:

```text id="z6qjlwm"
|d|
```

---

# ৩. `|Δ expression|`

## কেন?

Actual expression change কত বড় সেটা দেখার জন্য।

---

## Formula

\Delta expression = \mu_{CKD}-\mu_{Control}

---

## Example

| Gene  | Difference |
| ----- | ---------- |
| TLR7  | +8         |
| GeneX | +0.2       |

TLR7 stronger biomarker হতে পারে।

---

# তাহলে পুরো Rank Score কী করছে?

এটা এমন gene খুঁজছে যেগুলো:

✅ highly significant
✅ বড় effect size
✅ বড় expression change

---

# কেন multiplication করা হয়েছে?

কারণ তিনটার একটাও weak হলে score কমে যাবে।

---

## Example

ধরো:

| Gene | FDR    | Effect | Expression |
| ---- | ------ | ------ | ---------- |
| A    | strong | strong | strong     |
| B    | strong | weak   | strong     |

Gene A এর score বেশি হবে।

---

# এরপর Final Formula

## Combined Biomarker Score

\text{combined biomarker score}=\text{rank score}\times(1+\text{CKD specificity score})

---

# এখানে specificity কেন add করা হয়েছে?

কারণ:

```text id="m1g4e9"
সব kidney injury gene CKD-specific না
```

কিছু gene:

* AKI তেও বাড়ে
* অন্য disease এইও বাড়ে

তাই CKD-specific gene দরকার।

---

# CKD Specificity Score কী?

এটা measure করে:

```text id="ujwjlwm"
gene CKD তে uniquely বেশি কিনা
```

---

# Example

| Gene | CKD  | AKI |
| ---- | ---- | --- |
| TLR7 | high | low |

→ good CKD biomarker

---

# কেন `(1 + specificity)` ?

যদি specificity না থাকে:

```text id="x0vjlwm"
score × 1
```

মানে:

* penalty নেই
* boost ও নেই

যদি specificity high হয়:

```text id="j9dylo"
score × বড় number
```

→ CKD-specific gene উপরে উঠে আসে।

---

# Biological Logic

পুরো formula design করা হয়েছে যেন final biomarker হয়:

| Requirement              | Formula Part      |
| ------------------------ | ----------------- |
| statistically reliable   | −log10(FDR)       |
| biologically বড় change   | Cohen’s d         |
| expression difference বড় | Δ expression      |
| CKD-specific             | specificity score |

---

# Final Intuition

```text id="ljwmm2"
Strong Biomarker =
Reliable
×
Large Effect
×
Large Change
×
CKD Specific
```
